# Construction du dictionnaire de synonymes CIM-10

Ce notebook charge les différentes sources de synonymes CIM-10, les fusionne et les déduplique pour produire un fichier `synonymes.csv` utilisable dans le notebook d'évaluation `Evaluation_Synonymes_CIM10.ipynb`.

Sources incluses :
- **Dictionnaire AP-HP (Hector)** — plusieurs onglets thématiques
- **Orphanet** — classification des maladies rares
- **CépiDc** — dictionnaire des causes de décès
- **Inclusions officielles CIM-10** — notes d'inclusion de la classification

## 1. Imports et montage du Drive

In [1]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 2. Chargement des sources
### 2.1 Dictionnaire AP-HP (Hector)

On charge chaque onglet du fichier Excel séparément et on les concatène.
On ajoute une colonne `source` avec le nom de l'onglet pour tracer l'origine de chaque synonyme.
L'onglet "Cim Analytique" (source A) est exclu car il s'agit de la liste analytique ICD-Définition, moins fiable que les autres sources.

In [2]:
PATH_DATA = "/content/drive/MyDrive/Referentials/ICD/"

# Onglets à charger avec leur nom de source
# L'onglet "Cim Analytique" (source A = ICD-Définition) est volontairement exclu
onglets_hector = {
    "Cim Alphabétique"    : "Dictionnaire AP-HP Cim Alphabétique",
    "Thesam"              : "Dictionnaire AP-HP Thesam",
    "Dermatologie"        : "Dictionnaire AP-HP Dermatologie",
    "Endocrinologie"      : "Dictionnaire AP-HP Endocrinologie",
    "GRONES"              : "Dictionnaire AP-HP GRONES",
    "Troubles métaboliques": "Dictionnaire AP-HP Troubles métaboliques",
    "Néphrologie"         : "Dictionnaire AP-HP Néphrologie",
    "Ophtalmo"            : "Dictionnaire AP-HP Ophtalmo",
    "Orphanet"            : "Dictionnaire AP-HP Orphanet",
    "Rhumatologie"        : "Dictionnaire AP-HP Rhumatologie",
    "Germes"              : "Dictionnaire AP-HP Germes",
    "SRLF"                : "Dictionnaire AP-HP SRLF",
}

dfs_hector = []


In [3]:
for sheet_name, nom_source in onglets_hector.items():
    tmp = pd.read_excel(
        PATH_DATA + "CIM_APHP_2019/Dictionnaire_Hector_MAJ062019.xlsx",
        sheet_name=sheet_name,
        names=["libelle", "source_excel", "code", "autre_code"]
    )
    # On remplace la colonne source du fichier Excel par notre nom de source standardisé
    tmp["source"] = nom_source
    dfs_hector.append(tmp)

df_hector = pd.concat(dfs_hector, axis=0, ignore_index=True)


In [4]:
# On ne garde que les colonnes utiles
df_hector = df_hector[["code", "libelle", "source"]].rename(columns={"libelle": "synonyme"})

print(f"Nb lignes Hector : {len(df_hector)}")
print(df_hector.head())

Nb lignes Hector : 81867
   code                                           synonyme  \
0  A000  Choléra (asiatique) (épidémique) (malin), clas...   
1  A000  Choléra (asiatique) (épidémique) (malin), vibr...   
2  A001   Choléra (asiatique) (épidémique) (malin), el tor   
3  A001  Choléra (asiatique) (épidémique) (malin), vibr...   
4  A001  Entérite (aiguë) (diarrhéique) (épidémique) (p...   

                                source  
0  Dictionnaire AP-HP Cim Alphabétique  
1  Dictionnaire AP-HP Cim Alphabétique  
2  Dictionnaire AP-HP Cim Alphabétique  
3  Dictionnaire AP-HP Cim Alphabétique  
4  Dictionnaire AP-HP Cim Alphabétique  


### 2.2 Orphanet

Chargement du fichier Orphanet au format pickle (déjà prétraité).

In [5]:
chemin_orphanet = "/content/drive/MyDrive/Colab_Notebooks/Serenic_M/Dictionnaire/orphanet.pkl"

df_orphanet = pd.read_pickle(chemin_orphanet)

# On standardise les colonnes et on ajoute la source
df_orphanet = df_orphanet[["code", "extract"]].rename(columns={"extract": "synonyme"})
df_orphanet["source"] = "Orphanet"

print(f"Nb lignes Orphanet : {len(df_orphanet)}")
print(df_orphanet.head())

Nb lignes Orphanet : 13743
   code                                           synonyme    source
0  Q773  Syndrome de dysplasie épiphysaire multiple-mac...  Orphanet
1  G938                                Maladie d'Alexander  Orphanet
2  Q773  Syndrome de dysplasie épiphysaire multiple-min...  Orphanet
3  E771                                  Alpha-mannosidose  Orphanet
4  Q773  Syndrome de dysplasie épiphysaire multiple-dys...  Orphanet


### 2.3 CépiDc

Chargement du dictionnaire CépiDc (causes de décès, 2015).

In [6]:
chemin_cepidc = "/content/drive/MyDrive/Referentials/ICD/CepiDc_Dictionnaire2015.csv"

df_cepidc = pd.read_csv(chemin_cepidc, sep=";")

print("Colonnes disponibles :", df_cepidc.columns.tolist())
print(df_cepidc.head())

Colonnes disponibles : ['DiagnosisText', 'Icd1', 'IcdC', 'Icd2', 'CodingFlag', 'Likelihood', 'Prompt', 'YearStart', 'YearEnd', 'DateIn', 'UserIn', 'DateOut', 'UserOut', 'Comments', 'OKForMultipleCodes', 'FixForDT', 'DesignedByDT', 'NoStand1ByDT', 'NoStand2ByDT']
          DiagnosisText  Icd1 IcdC Icd2 CodingFlag  Likelihood  Prompt  \
0  2 greffes hépatiques  Z944  NaN  NaN        NaN         NaN     NaN   
1                47 XXY  Q980  NaN  NaN        NaN         NaN     NaN   
2                47 XYY  Q985  NaN  NaN        NaN         NaN     NaN   
3          à déterminer   R97  NaN  NaN        NaN         NaN     NaN   
4         AA thoracique  I712  NaN  NaN        NaN         NaN     NaN   

   YearStart  YearEnd                   DateIn UserIn  \
0       2011     2025  2011-05-02 00:00:00.000  User1   
1       2011     2025  2011-05-02 00:00:00.000  User1   
2       2011     2025  2011-05-02 00:00:00.000  User1   
3       2011     2025  2011-05-02 00:00:00.000  User1   
4      

In [7]:
# Le synonyme est dans 'DiagnosisText', le code principal dans 'Icd1'
# On ignore les lignes sans code Icd1
df_cepidc = df_cepidc[["DiagnosisText", "Icd1"]].rename(
    columns={"DiagnosisText": "synonyme", "Icd1": "code"}
)

# Suppression des lignes sans code
df_cepidc = df_cepidc.dropna(subset=["code"]).reset_index(drop=True)

df_cepidc["source"] = "CépiDc"

print(f"Nb lignes CépiDc : {len(df_cepidc)}")
print(df_cepidc.head())

Nb lignes CépiDc : 147340
               synonyme  code  source
0  2 greffes hépatiques  Z944  CépiDc
1                47 XXY  Q980  CépiDc
2                47 XYY  Q985  CépiDc
3          à déterminer   R97  CépiDc
4         AA thoracique  I712  CépiDc


### 2.4 Inclusions officielles CIM-10

In [8]:
chemin_inclusions = "/content/drive/MyDrive/Referentials/ICD/CIM_OFS_SW_2006/inclusions_v2.xlsx"

df_inclusions = pd.read_excel(chemin_inclusions)

print("Colonnes disponibles :", df_inclusions.columns.tolist())
print(df_inclusions.head())

Colonnes disponibles : ['code', 'prefix', 'extract', 'definition', 'label']
   code prefix                                   extract  \
0  A000    NaN                         choléra classique   
1  A001    NaN                            choléra El Tor   
2  A010    NaN          infection due à Salmonella typhi   
3  A014    NaN  infection due à Salmonella paratyphi SAI   
4  A022    NaN                 pneumopathie à Salmonella   

                                      definition  label  
0  Choléra à Vibrio cholerae 01, biovar cholerae      1  
1    Choléra à Vibrio cholerae 01, biovar El Tor      1  
2                                Fièvre typhoïde      1  
3                   Paratyphoïde, sans précision      1  
4               Infection localisée à Salmonella      1  


In [9]:
# Le synonyme est dans 'extract', le code dans 'code'
df_inclusions = df_inclusions[["code", "extract"]].rename(columns={"extract": "synonyme"})

# Suppression des lignes sans synonyme ou sans code
df_inclusions = df_inclusions.dropna(subset=["code", "synonyme"]).reset_index(drop=True)

df_inclusions["source"] = "Inclusions CIM-10"

print(f"Nb lignes Inclusions CIM-10 : {len(df_inclusions)}")
print(df_inclusions.head())

Nb lignes Inclusions CIM-10 : 7246
   code                                  synonyme             source
0  A000                         choléra classique  Inclusions CIM-10
1  A001                            choléra El Tor  Inclusions CIM-10
2  A010          infection due à Salmonella typhi  Inclusions CIM-10
3  A014  infection due à Salmonella paratyphi SAI  Inclusions CIM-10
4  A022                 pneumopathie à Salmonella  Inclusions CIM-10


## 3. Fusion et déduplication

On concatène les 4 sources en un seul DataFrame.
Lors de la déduplication, si un même synonyme apparaît dans plusieurs sources,
on conserve une seule ligne mais on concatène les noms des sources séparés par un espace.
Exemple : un synonyme présent dans CépiDc, Hector B et Orphanet donnera :
`source = "CépiDc / Dictionnaire AP-HP Cim Alphabétique / Orphanet"`

In [10]:
# Concaténation des 4 sources
df_total = pd.concat(
    [df_hector, df_orphanet, df_cepidc, df_inclusions],
    axis=0,
    ignore_index=True
)

print(f"Nb lignes avant déduplication : {len(df_total)}")

Nb lignes avant déduplication : 250196


In [11]:
# Suppression des lignes sans synonyme ou sans code
df_total = df_total.dropna(subset=["code", "synonyme"]).reset_index(drop=True)

print(f"Nb lignes après suppression des NaN : {len(df_total)}")

Nb lignes après suppression des NaN : 250196


In [12]:
# Déduplication : pour chaque synonyme identique, on garde une seule ligne
# et on concatène les sources séparées par ' / '
df_dedup = (
    df_total.groupby("synonyme", as_index=False)
    .agg(
        code=("code", "first"),       # on garde le premier code trouvé
        source=("source", lambda x: " / ".join(x.unique()))  # on concatène les sources
    )
)

print(f"Nb lignes après déduplication : {len(df_dedup)}")
print(df_dedup.head())

Nb lignes après déduplication : 229739
                                            synonyme  code  \
0                              'Cone-rod' dystrophie  H355   
1  'Hypoplasie ''cartilage-cheveux''-like sans hy...  Q785   
2                              'Syndrome der(22)t(11  Q926   
3  (broncho)pneumopathie grippale, autre virus gr...  J100   
4  (broncho)pneumopathie virale, sans précision o...  J110   

                        source  
0  Dictionnaire AP-HP Orphanet  
1  Dictionnaire AP-HP Orphanet  
2  Dictionnaire AP-HP Orphanet  
3            Inclusions CIM-10  
4            Inclusions CIM-10  


In [13]:
# Affichage des différents types de combinaisons de sources
# (pour voir combien de synonymes viennent d'une seule source, de deux sources, etc.)

print("Répartition par combinaison de sources :")
print("-" * 70)

combinaisons = df_dedup["source"].value_counts()

for source, count in combinaisons.items():
    print(f"  {source:<55} : {count:>7} synonymes")

print("-" * 70)
print(f"  Nb de combinaisons distinctes : {len(combinaisons)}")

Répartition par combinaison de sources :
----------------------------------------------------------------------
  CépiDc                                                  :  130611 synonymes
  Dictionnaire AP-HP Cim Alphabétique                     :   45193 synonymes
  Dictionnaire AP-HP Thesam                               :   21220 synonymes
  Orphanet                                                :   10340 synonymes
  Dictionnaire AP-HP Orphanet                             :    8232 synonymes
  Inclusions CIM-10                                       :    6552 synonymes
  Dictionnaire AP-HP Orphanet / Orphanet                  :    2039 synonymes
  Dictionnaire AP-HP Dermatologie                         :    1783 synonymes
  Dictionnaire AP-HP Rhumatologie                         :    1031 synonymes
  Dictionnaire AP-HP Néphrologie                          :     562 synonymes
  Dictionnaire AP-HP Ophtalmo                             :     435 synonymes
  CépiDc / Inclusions CIM-10  

## 4. Filtrage des codes du chapitre XXI (codes Z)

On supprime tous les synonymes dont le code commence par 'Z' (chapitre XXI de la CIM-10).

In [14]:
nb_avant = len(df_dedup)

# Suppression de toutes les lignes dont le code commence par Z
df_dedup = df_dedup[~df_dedup["code"].astype(str).str.startswith("Z")].reset_index(drop=True)

nb_apres = len(df_dedup)

print(f"Nb lignes avant filtrage codes Z : {nb_avant}")
print(f"Nb lignes après filtrage codes Z : {nb_apres}")
print(f"Nb lignes supprimées             : {nb_avant - nb_apres}")

Nb lignes avant filtrage codes Z : 229739
Nb lignes après filtrage codes Z : 220891
Nb lignes supprimées             : 8848


## 5. Export du dictionnaire final

On exporte le dictionnaire dédupliqué et filtré dans un fichier `synonymes.csv`
qui sera utilisé dans le notebook d'évaluation `Evaluation_Synonymes_CIM10.ipynb`.

Le fichier contient 3 colonnes :
- `code` : code CIM-10
- `synonyme` : terme médical
- `source` : source(s) d'origine séparées par ' / '

In [15]:
chemin_export = "/content/drive/MyDrive/Colab_Notebooks/Serenic_M/Recherche_Synonymes_CIM10/synonymes.csv"
# chemin_export = "C:/Users/ton_nom/Documents/Recherche_Synonymes_CIM10/synonymes.csv"  # chemin local

df_dedup[["code", "synonyme", "source"]].to_csv(
    chemin_export,
    sep=";",
    index=False,
    encoding="utf-8-sig"  # utf-8-sig pour compatibilité Excel
)

print(f" Fichier exporté : {chemin_export}")
print(f"   Nb synonymes    : {len(df_dedup)}")
print(f"   Nb codes uniques: {df_dedup['code'].nunique()}")

 Fichier exporté : /content/drive/MyDrive/Colab_Notebooks/Serenic_M/Recherche_Synonymes_CIM10/synonymes.csv
   Nb synonymes    : 220891
   Nb codes uniques: 9895
